### Box Office Analysis for Strategic Movie Production
This notebook contains a full end-to-end exploratory data analysis (EDA) combining IMDB and Box Office Mojo data to inform a new movie studio's production strategy.
It includes data loading, cleaning, merging (exact + fuzzy matching), EDA visualizations, and three concrete business recommendations.

# Importing Data and Settings

In [4]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from difflib import get_close_matches
plt.rcParams['figure.figsize'] = (10,6)
plt.rcParams['font.size'] = 12

## 2) Paths and helper functions

In [7]:
import os
import zipfile

# Set up data directory
BASE_DIR = os.path.dirname(os.path.abspath('group_4_project.ipynb'))
DATA_DIR = os.path.join(BASE_DIR, 'zippedData')
IMDB_DB = os.path.join(DATA_DIR, 'im.db')
BOM_FILE = os.path.join(DATA_DIR, 'bom.movie_gross.csv.gz')
OUTPUT_DIR = os.path.join(DATA_DIR, 'eda_outputs')
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Extract im.db if it's still zipped
im_db_zip = os.path.join(DATA_DIR, 'im.db.zip')
if os.path.exists(im_db_zip) and not os.path.exists(IMDB_DB):
    print('Extracting im.db.zip...')
    with zipfile.ZipFile(im_db_zip, 'r') as zip_ref:
        zip_ref.extractall(DATA_DIR)
    print('Done extracting.')

def clean_money_column(s):
    return s.astype(str).str.replace(r'[$,]', '', regex=True).replace('', np.nan)

def save_fig(fig, fname):
    path = os.path.join(OUTPUT_DIR, fname)
    fig.savefig(path, bbox_inches='tight')

Extracting im.db.zip...
Done extracting.


## 3) Load data

In [8]:
print('Loading IMDB database...')
print(f'Database path: {IMDB_DB}')
print(f'Database exists: {os.path.exists(IMDB_DB)}')

conn = sqlite3.connect(IMDB_DB)
movie_basics = pd.read_sql('SELECT * FROM movie_basics;', conn)
movie_ratings = pd.read_sql('SELECT * FROM movie_ratings;', conn)
conn.close()

print('Loading BOM...')
bom = pd.read_csv(BOM_FILE, compression='gzip', low_memory=False)
print('Done. Rows:', len(movie_basics), 'IMDB basics; ', len(bom), 'BOM rows')

Loading IMDB database...
Database path: c:\Users\kanja\OneDrive\Documents\group_4_project_phase_2\zippedData\im.db
Database exists: True
Loading BOM...
Done. Rows: 146144 IMDB basics;  3387 BOM rows
